In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np

crashes = pd.read_csv("../data/processed/crashes_cleaned.csv")
segment_risk = pd.read_csv("../data/processed/segment_risk.csv")
roads = gpd.read_file("../data/external/nyc_roads.gpkg")

print(f"Crashes: {len(crashes):,}")
print(f"Segments with risk: {len(segment_risk):,}")
print(f"Road segments: {len(roads):,}")

Crashes: 461,425
Segments with risk: 63,265
Road segments: 122,272


In [3]:
# convert physicalid to same type for merging
roads['physicalid'] = roads['physicalid'].astype(str)
segment_risk['physicalid'] = segment_risk['physicalid'].astype(str)

# drop geometry before merge - this is what was causing the freeze
roads_df = pd.DataFrame(roads.drop(columns='geometry'))

# merge road attributes onto risk segments
df = segment_risk.merge(
    roads_df[['physicalid', 'street_name', 'rw_type', 'posted_speed',
           'number_total_lanes', 'number_travel_lanes', 'trafdir',
           'segmentlength', 'streetwidth', 'bike_lane', 'snow_priority']],
    on='physicalid', how='left'
)

print(f"Shape: {df.shape}")
print("\nMissing values:")
print(df.isnull().sum()[df.isnull().sum() > 0])

Shape: (63665, 16)

Missing values:
posted_speed            2502
number_total_lanes      1002
number_travel_lanes      990
segmentlength            569
streetwidth             1005
bike_lane              53422
snow_priority            880
dtype: int64


In [ ]:
import numpy as np
import pandas as pd

# reload the spatial join result with physicalid attached to each crash
crashes = pd.read_csv("../data/processed/crashes_cleaned.csv")

# reload joined data - we need to save it first from spatial_join notebook
# for now aggregate temporal from crashes we have
crashes['crash_datetime'] = pd.to_datetime(crashes['crash_datetime'])
crashes['hour'] = crashes['crash_datetime'].dt.hour
crashes['month'] = crashes['crash_datetime'].dt.month
crashes['dayofweek'] = crashes['crash_datetime'].dt.dayofweek

crashes['hour_sin'] = np.sin(2 * np.pi * crashes['hour'] / 24)
crashes['hour_cos'] = np.cos(2 * np.pi * crashes['hour'] / 24)
crashes['is_rush_hour'] = crashes['hour'].between(7,9) | crashes['hour'].between(17,19)
crashes['is_weekend'] = crashes['dayofweek'] >= 5

print("Temporal features computed.")
print(crashes[['hour','is_rush_hour','is_weekend']].head())

KeyError: 'physicalid'